# Yeti Travel — Customer Churn Prediction

**Machine Learning & Business Applications — Group Project**

## Business Context

Yeti Travel is a travel agency specialised in **school trips**. Over the past year
the company has registered a concerning **drop in sales, especially among repeat customers**.
Since most of the revenue comes from loyal clients, this threatens the continuity of the
business.

Our goal is to:

1. Build a **predictive model** that estimates the probability that a client (a school) will
   come back the following year (`Retained = 1`) or churn (`Retained = 0`).
2. Identify the **most informative features** of loyal customers, so that the agency can
   design targeted retention strategies (personalised offers, tighter relationships with
   high-risk schools, etc.).

## Data

The data is split across three departments, each with its own snapshot at the end of
"year 1":

- **Sales** — trip-level commercial info (program, grades, travel mode, dates, passengers).
- **Finance** — financial info (tuition, insurance take-up, revenue, payment modality).
- **CRM** — school-level info (school type, size, region, parent meetings, income level).

The target `Retained` lives in the Sales dataset.


## 1. Setup and imports

We use the standard data-science stack: `pandas`/`numpy` for data handling,
`matplotlib`/`seaborn` for visualisation, and `scikit-learn` for the full ML pipeline
(preprocessing, cross-validation, models).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, classification_report

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 2. Loading the data

We load both the **training (`_model`)** and the **test (`_test`)** versions of the
three datasets. The training set is labelled (`Retained` is known) and will be used
to fit the model. The test set is unlabelled — our model's predictions on it are the
deliverable.


In [2]:
path = 'data/'

datasets = [
    ('sales',   'sales_model.csv',   'sales_test.csv'),
    ('finance', 'finance_model.csv', 'finance_test.csv'),
    ('crm',     'crm_model.csv',     'crm_test.csv'),
]

# Load everything into a dict for later reuse
data = {}
for name, train_file, test_file in datasets:
    data[f'{name}_train'] = pd.read_csv(path + train_file)
    data[f'{name}_test']  = pd.read_csv(path + test_file)

# Quick shape summary (like your second script)
print("Training shapes:")
for name, _, _ in datasets:
    print(f"  {name}_train: {data[f'{name}_train'].shape}")
print("\nTest shapes:")
for name, _, _ in datasets:
    print(f"  {name}_test:  {data[f'{name}_test'].shape}")

# Detailed overview (like your first script) for every dataframe
for key, df in data.items():
    print(f"\n{'='*40}")
    print(f"{key}: {df.shape}")
    print(df.head(3).to_string())
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        print("\nMissing values:")
        print(missing)
    else:
        print("\nNo missing values.")

Training shapes:
  sales_train: (4153, 24)
  finance_train: (4151, 12)
  crm_train: (4148, 20)

Test shapes:
  sales_test:  (630, 24)
  finance_test:  (630, 12)
  crm_test:  (630, 20)

sales_train: (4153, 24)
  ID_SALES Program_Code  From_Grade  To_Grade Group_State  Days Travel_Type Departure_Date Return_Date   Early_RPL  Latest_RPL  Cancelled_Pax  Total_Discount_Pax Initial_System_Date SPR_Product_Type  FPP  Total_Pax DepartureMonth GroupGradeTypeLow GroupGradeTypeHigh GroupGradeType MajorProgramCode FPP_to_School_enrollment  Retained
0  CC1387A           CC        10.0      10.0          CA    24           A     04/07/2019  05/01/2019  05/11/2018  04/10/2018              2                   1          04/01/2018       East Coast   26         29        January                 K               High        K->High                C        0,126429354314411         1
1   CC139A           CC         5.0       6.0          CA    18           A     04/30/2019  05/18/2019  03/05/2018  03/06/2

## 3. Merging the three datasets

The three datasets share the same underlying trip, but each department uses its
**own ID format**:

| Dataset  | Example ID  |
|----------|-------------|
| Sales    | `CC1040A`   |
| Finance  | `CP1040`    |
| CRM      | `01040K`    |

The common element is the **numeric part** (`1040`). We extract it with a regex
and use it as the join key.

⚠️ **Important caveat we ran into:** if we extract the digits as strings, the
CRM IDs preserve their leading zeros (`"01040"` vs `"1040"`) and the join silently
fails — producing a merged dataset where all CRM columns are `NaN`. Casting the
extracted number to `int` solves it.

We also drop duplicate CRM rows (a single school can appear multiple times, but
we only need it once) and use a `left` join from Sales so we keep every trip —
even those without a CRM match, which we'll handle later via imputation.


In [3]:
def merge_datasets(sales, finance, crm):
    """Merge the three source datasets on the numeric trip ID."""
    # Extract digits and cast to int (important: handles leading zeros in CRM)
    sales   = sales.copy()
    finance = finance.copy()
    crm     = crm.copy()

    # handle leading zeros in some _id columns
    sales['trip_id']   = sales['ID_SALES'].str.extract(r'(\d+)')[0].astype(int)
    finance['trip_id'] = finance['ID_FINANCE'].str.extract(r'(\d+)')[0].astype(int)
    crm['trip_id']     = crm['ID_CRM'].str.extract(r'(\d+)')[0].astype(int)

    # A school can have multiple CRM rows — keep the first one
    crm = crm.drop_duplicates(subset='trip_id', keep='first')

    # Left join: keep all trips from sales, enrich with finance and CRM
    df = (
        sales
        .merge(finance, on='trip_id', how='left')
        .merge(crm,     on='trip_id', how='left')
        .drop(columns='trip_id')
    )
    return df


We can now unpack the dictionary we created and split in "train" and "test" datasets. Once we have done that, we can merge the training datasets

In [5]:
# Unpack the dictionary dataset and split 
sales_train,   sales_test   = data['sales_train'],   data['sales_test']
finance_train, finance_test = data['finance_train'], data['finance_test']
crm_train,     crm_test     = data['crm_train'],     data['crm_test']

# Merge both splits
df_train = merge_datasets(sales_train, finance_train, crm_train)
df_test  = merge_datasets(sales_test,  finance_test,  crm_test)

# Preview shapes of merged 
print(f"Merged train shape: {df_train.shape}")
print(f"Merged test  shape: {df_test.shape}")

# preview
df_train.head(10)

Merged train shape: (4153, 56)
Merged test  shape: (630, 56)


,ID_SALES,Program_Code,From_Grade,To_Grade,Group_State,Days,Travel_Type,Departure_Date,Return_Date,Early_RPL,...,SPR_New_Existing,NumberOfMeetingswithParents,FirstMeeting,LastMeeting,DifferenceTraveltoFirstMeeting,DifferenceTraveltoLastMeeting,SchoolGradeTypeLow,SchoolGradeTypeHigh,SchoolGradeType,SchoolSizeIndicator
0,CC1387A,CC,10.0,10.0,CA,24,A,04/07/2019,05/01/2019,05/11/2018,...,1.0,2.0,05/10/2018,09/04/2018,332.0,215.0,Middle,Middle,Middle->Middle,S
1,CC139A,CC,5.0,6.0,CA,18,A,04/30/2019,05/18/2019,03/05/2018,...,0.0,1.0,09/04/2018,08/29/2018,238.0,244.0,Middle,Middle,Middle->Middle,S
2,CC1600A,CC,7.0,7.0,OR,13,A,03/17/2019,03/30/2019,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CC1701A,CC,8.0,8.0,CA,24,A,05/08/2019,06/01/2019,02/07/2018,...,0.0,1.0,08/16/2018,08/27/2018,265.0,254.0,Middle,Middle,Middle->Middle,S
4,CC1773A,CC,7.0,7.0,OR,13,A,03/17/2019,03/30/2019,NaN,...,1.0,1.0,09/18/2018,09/09/2018,180.0,189.0,Middle,Middle,Middle->Middle,S
5,CC1830A,CC,7.0,8.0,CA,30,A,06/23/2019,07/23/2019,02/25/2018,...,0.0,1.0,09/16/2018,09/15/2018,280.0,281.0,Middle,Middle,Middle->Middle,S
6,CC1836A,CC,9.0,9.0,CA,33,A,03/07/2019,04/09/2019,02/16/2018,...,1.0,1.0,09/09/2018,09/04/2018,179.0,184.0,Middle,Middle,Middle->Middle,L
7,CC242A,CC,7.0,7.0,CA,28,A,03/09/2019,04/06/2019,02/22/2018,...,0.0,1.0,09/06/2018,08/25/2018,184.0,196.0,Middle,Middle,Middle->Middle,L
8,CC2531A,CC,9.0,10.0,CA,11,A,05/03/2019,05/14/2019,03/13/2018,...,1.0,1.0,08/26/2018,09/07/2018,250.0,238.0,Middle,Middle,Middle->Middle,S
9,CC2637A,CC,10.0,10.0,CA,24,A,04/29/2019,05/23/2019,03/27/2018,...,0.0,2.0,04/22/2018,08/22/2018,372.0,250.0,Middle,Middle,Middle->Middle,S


## 4. Exploratory Data Analysis

Before modelling, we look at the data to build intuition. The main questions are:

1. **Is the target balanced?** This drives metric choice.
2. **Where are the missing values?** This drives imputation strategy.
3. **Which features look predictive?** This helps feature engineering.


### 4.1 Class balance

The class balance tells us how common retention is. If the dataset is heavily imbalanced,
accuracy becomes a misleading metric and we should rely on **ROC-AUC** and **F1** instead.


In [ ]:
balance = train['Retained'].value_counts()
balance_pct = train['Retained'].value_counts(normalize=True).round(3)
print("Counts:")
print(balance)
print("\nProportions:")
print(balance_pct)

fig, ax = plt.subplots(figsize=(6, 4))
balance.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_title('Class balance — Retained vs Churned')
ax.set_xticklabels(['Churned (0)', 'Retained (1)'], rotation=0)
ax.set_ylabel('Number of clients')
plt.tight_layout()
plt.show()

### 4.2 Missing values

Different columns have very different rates of missingness. A few patterns to notice:

- **`Early_RPL`** has many nulls — it's the first communication invitation date,
  and it's likely only recorded when the school reached a certain stage of the funnel.
- **CRM meeting dates** (`FirstMeeting`, `LastMeeting`, etc.) are missing for schools
  that held no meetings.
- **`Special_Pay`** is missing for most rows — we'll drop it.


In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(1)
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
print(missing_df.head(20))

fig, ax = plt.subplots(figsize=(10, 5))
missing.head(15).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 15 columns by missing count')
ax.set_ylabel('# missing')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4.3 Retention by key categorical variables

A quick way to screen for predictive features is to compute the retention rate
for each level of a categorical variable. Big differences across levels suggest
the variable carries information about churn.


In [ ]:
cat_features_to_check = ['Travel_Type', 'SPR_Product_Type', 'MajorProgramCode',
                         'DepartureMonth', 'School_Type', 'Region']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, cat_features_to_check):
    rates = train.groupby(col)['Retained'].mean().sort_values(ascending=False)
    rates.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'Retention rate by {col}')
    ax.set_ylabel('Retention rate')
    ax.set_ylim(0, 1)
    ax.axhline(train['Retained'].mean(), color='red', ls='--', alpha=0.5, label='overall')
    ax.legend()
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### 4.4 Numeric features vs target

For a few key numeric variables we look at their distribution split by target.
If the distributions differ, the feature likely carries signal.


In [ ]:
num_to_check = ['Tuition', 'Days', 'FPP', 'Total_Pax', 'Cancelled_Pax']

fig, axes = plt.subplots(1, len(num_to_check), figsize=(18, 4))
for ax, col in zip(axes, num_to_check):
    train.boxplot(column=col, by='Retained', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('Retained')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 5. Feature engineering

Now that we've looked at the raw data, we build features the models can actually
consume. Four things to do:

1. **Fix the finance decimal encoding.** Finance columns like `FRP_Take_up_percent_`
   were exported with European notation (`0,123` instead of `0.123`), so pandas reads
   them as strings. We replace the comma and cast to float.
2. **Parse dates** and derive informative time deltas:
   - `lead_time_days` — days between first system entry and departure. Long lead times
     indicate more organised clients.
   - `rpl_window_days` — duration of the enrolment campaign.
   - `departure_season` — a higher-level categorical summary of `DepartureMonth`.
3. **Build ratios** that normalise for group size:
   - `cancellation_rate` — fraction of registered passengers who cancelled.
   - `discount_rate` — share of the group that got a discount.
4. **Drop columns** that are either:
   - Pure identifiers (IDs),
   - Raw dates (already summarised by our engineered time-deltas),
   - Duplicates (`GroupGradeType` already encodes `Low` + `High`),
   - Near-empty (`Special_Pay`).


In [ ]:
def fix_comma_decimals(df, cols):
    """Replace ',' with '.' in numeric columns stored as strings, then cast to float."""
    for c in cols:
        if c in df.columns and df[c].dtype == object:
            df[c] = df[c].astype(str).str.replace(',', '.', regex=False)
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df


def engineer_features(df):
    df = df.copy()

    # 1. Fix European decimal commas in finance-related columns
    euro_cols = ['FRP_Take_up_percent_', 'EZ_Pay_Take_Up_Rate',
                 'FPP_to_PAX', 'FPP_to_School_enrollment']
    df = fix_comma_decimals(df, euro_cols)

    # 2. Parse date columns
    date_cols = ['Departure_Date', 'Return_Date', 'Early_RPL',
                 'Latest_RPL', 'Initial_System_Date']
    for c in date_cols:
        df[c] = pd.to_datetime(df[c], format='%m/%d/%Y', errors='coerce')

    # Derived time-delta features
    df['lead_time_days']  = (df['Departure_Date'] - df['Initial_System_Date']).dt.days
    df['rpl_window_days'] = (df['Latest_RPL']    - df['Early_RPL']).dt.days

    # Season from departure month
    month_map = {1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',
                 6:'Summer',7:'Summer',8:'Summer',9:'Fall',10:'Fall',
                 11:'Fall',12:'Winter'}
    df['departure_season'] = df['Departure_Date'].dt.month.map(month_map).fillna('Unknown')

    # 3. Ratio features (the +1 avoids division by zero)
    df['cancellation_rate'] = df['Cancelled_Pax'] / (df['Total_Pax'] + 1)
    df['discount_rate']     = df['Total_Discount_Pax'] / (df['Total_Pax'] + 1)

    # 4. Drop columns we no longer need
    drop_cols = [
        'ID_SALES', 'ID_FINANCE', 'ID_CRM',
        # Raw dates (now summarised by our engineered features)
        'Departure_Date', 'Return_Date', 'Early_RPL', 'Latest_RPL',
        'Initial_System_Date', 'FirstMeeting', 'LastMeeting', 'Deposit_Date',
        # Redundant combo columns (GroupGradeType already = Low + High)
        'GroupGradeTypeLow', 'GroupGradeTypeHigh',
        'SchoolGradeTypeLow', 'SchoolGradeTypeHigh',
        # Near-empty
        'Special_Pay',
    ]
    drop_cols += [c for c in df.columns if 'Unnamed' in str(c)]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)

# Separate the target from the features
y_train = train_fe.pop('Retained')

print(f"After feature engineering — train: {train_fe.shape}, test: {test_fe.shape}")
train_fe.head(3)

## 6. Preprocessing pipeline

We build a `ColumnTransformer` that applies the right treatment to each column type,
wrapped inside a `Pipeline` so that **every preprocessing step is learned on the
training folds only** during cross-validation — no data leakage.

- **Numeric columns**: median imputation (robust to outliers) + standard scaling
  (needed for Logistic Regression; harmless for trees).
- **Categorical columns**: most-frequent imputation + one-hot encoding with
  `handle_unknown='ignore'` (so unseen categories at prediction time don't crash).


In [ ]:
num_cols = train_fe.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = train_fe.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns ({len(num_cols)}): {num_cols}")
print(f"\nCategorical columns ({len(cat_cols)}): {cat_cols}")

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
])

## 7. Model comparison via cross-validation

We compare three progressively more complex models:

1. **Logistic Regression** — simple linear baseline. If a complex model can't beat this,
   the task is essentially linear and we should keep it simple.
2. **Random Forest** — non-linear, captures interactions, robust to outliers.
3. **Gradient Boosting** — usually the strongest classical tabular model.

Evaluation uses **5-fold stratified cross-validation** (stratified because we want each
fold to preserve the class balance) and two metrics:

- **ROC-AUC** — captures ranking quality, threshold-independent.
- **F1** — balances precision and recall for the positive class.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', model)])
    scores = cross_validate(
        pipe, train_fe, y_train, cv=cv,
        scoring=['roc_auc', 'f1'],
        return_train_score=False,
        n_jobs=-1
    )
    results[name] = {
        'ROC-AUC':     scores['test_roc_auc'].mean(),
        'ROC-AUC std': scores['test_roc_auc'].std(),
        'F1':          scores['test_f1'].mean(),
        'F1 std':      scores['test_f1'].std(),
    }

results_df = pd.DataFrame(results).T.round(4)
print(results_df)

best_name = results_df['ROC-AUC'].idxmax()
print(f"\nBest model by ROC-AUC: {best_name}")

## 8. Final model — fit on all training data

Now that Gradient Boosting (or whichever model wins) is the chosen one, we refit it
on the full training set (no more hold-out — we want to use every row of labelled data
before predicting on the test set).

We also extract **feature importances** from the tree-based model. This is what the
business team actually cares about: it tells us *why* churning customers churn.


In [ ]:
best_pipe = Pipeline([('prep', preprocessor), ('clf', models[best_name])])
best_pipe.fit(train_fe, y_train)

# Extract one-hot-encoded feature names so we can label importances correctly
if hasattr(models[best_name], 'feature_importances_'):
    ohe_cats = (
        best_pipe.named_steps['prep']
                 .named_transformers_['cat']
                 .named_steps['encoder']
                 .get_feature_names_out(cat_cols).tolist()
    )
    feature_names = num_cols + ohe_cats

    importances = pd.Series(
        best_pipe.named_steps['clf'].feature_importances_,
        index=feature_names
    ).sort_values(ascending=False)

    print("Top 15 features:")
    print(importances.head(15).round(4))

    fig, ax = plt.subplots(figsize=(9, 6))
    importances.head(15).sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Top 15 feature importances — {best_name}')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

## 9. Generating predictions for the test set

Final step: apply the trained pipeline to the test set and save predictions in
the format required by the assignment (two columns: `ID_SALES` and `Retained`).


In [ ]:
# We dropped ID_SALES during feature engineering — grab it back from the raw test set
test_ids = test['ID_SALES'].copy()

test_preds = best_pipe.predict(test_fe)

submission = pd.DataFrame({
    'ID_SALES': test_ids,
    'Retained': test_preds
})

submission.to_csv('predictions.csv', index=False)

print(f"Saved {len(submission)} predictions to predictions.csv")
print("\nPrediction distribution:")
print(submission['Retained'].value_counts())
print("\nFirst 10 predictions:")
print(submission.head(10))

## 10. Business takeaways

This section is what will drive the presentation's narrative. Based on the feature
importances, we can profile churn risk and suggest concrete retention levers:

- **High-importance time-related features** (`lead_time_days`, `rpl_window_days`) suggest
  that **how early and for how long a client engages** with the agency matters. Agents
  could actively nurture clients with short RPL windows — they're at higher risk.
- **Finance features** (`FRP_Take_up_percent_`, `EZ_Pay_Take_Up_Rate`) reflect the
  client's financial commitment. Low take-up on insurance or direct debit may indicate
  weaker attachment to the service.
- **CRM features** like `NumberOfMeetingswithParents` and `Parent_Meeting_Flag` suggest
  that **human contact matters**. Schools with more parent meetings may be more satisfied
  and hence more likely to return.
- **Operational features** (`cancellation_rate`, `Days`, `Tuition`) capture the intrinsic
  trip experience — high cancellations or outlier pricing may correlate with churn.

**Recommended actions for Yeti Travel:**

1. **Early-warning dashboard** — flag accounts where the model's predicted churn probability
   exceeds a threshold, and route them to a dedicated retention agent.
2. **Parent meetings as a retention lever** — actively promote meetings for schools with
   zero recorded meetings.
3. **Customer satisfaction tracking** — as Angela originally hinted, the biggest blind spot
   is the lack of direct satisfaction data. A simple post-trip survey would add powerful
   signal to future iterations of the model.
